# 03 · Statistical and Hypothesis Analysis

**Purpose:** Retain the original income, sex, education, geographic, and time-trend tests, including assumption checks, post-hoc analysis, effect size, and interpretation.

This notebook preserves the substantive code and analytical sequence from the original completed project. Changes are limited to portable file paths, organization, and explanatory markdown.


## Reproducible setup

The project-relative paths below replace the original Google Colab mount so the notebook runs consistently in VS Code, Jupyter, or GitHub Codespaces.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "cdc_brfss_nutrition_activity_obesity_2011_2024.csv"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "obesity_analysis_ready.csv"

# Each analysis notebook is independently runnable from the project root or notebooks folder.
clean_df = pd.read_csv(PROCESSED_DATA, low_memory=False)
data = clean_df.copy()
print(f"Loaded {len(clean_df):,} analysis-ready rows covering {clean_df.YearStart.min()}–{clean_df.YearStart.max()}.")


### 3.2 Statistical Analysis

## 3.1 Income differences

**Why this test is used:** One-way ANOVA evaluates whether average prevalence differs across more than two groups. A significant result is followed by practical interpretation and, where appropriate, post-hoc testing.


In [ ]:
from scipy.stats import f_oneway

# 1. Group data by Income category
income_groups = clean_df.groupby("Income")["Data_Value"]

# 2. Create list of arrays (each income group)
groups = [group.values for name, group in income_groups]

# 3. Perform One-Way ANOVA
anova_stat, anova_p = f_oneway(*groups)

print("One-Way ANOVA Result (Income Differences)")
print(f"F-statistic = {anova_stat:.4f}")
print(f"p-value = {anova_p:.8f}\n")

# 4. Hypothesis Testing
alpha = 0.05

print("Hypothesis:")
print("H0: Mean obesity is the same across all income groups")
print("H1: At least one income group has a different mean obesity\n")

if anova_p < alpha:
    print("Conclusion:")
    print("Reject H0. There is a statistically significant difference in obesity prevalence across income groups.\n")
else:
    print("Conclusion:")
    print("Fail to reject H0. There is no statistically significant evidence that obesity prevalence differs across income groups.\n")

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Income column is categorical and clean
clean_df = clean_df.dropna(subset=["Income", "Data_Value"])

# Define the order of income groups for better visualization
income_order = [
    "Less than $15,000",
    "$15,000 - $24,999",
    "$25,000 - $34,999",
    "$35,000 - $49,999",
    "$50,000 - $74,999",
    "$75,000 or greater"
]

# Create boxplot using Seaborn
plt.figure(figsize=(12, 7))
sns.boxplot(x="Income", y="Data_Value", data=clean_df, order=income_order, palette="viridis")

plt.title("Obesity Distribution Across Income Groups", fontsize=16)
plt.xlabel("Income Group", fontsize=12)
plt.ylabel("Obesity Percentage", fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**What this code examines:** Compares survey sample size with prevalence estimates to check whether estimate level appears related to coverage.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress

# Ensure national_trend is available from previous steps
national_trend = clean_df.groupby("YearStart")["Data_Value"].mean()

# Perform linear regression to get slope and intercept
slope, intercept, r_value, p_value, std_err = linregress(
    national_trend.index,
    national_trend.values
)

# Plot the actual data points
plt.figure(figsize=(10, 6))
plt.scatter(national_trend.index, national_trend.values, label='Actual Data', color='skyblue')

# Plot the linear regression line
reg_line = slope * national_trend.index + intercept
plt.plot(national_trend.index, reg_line, color='red', linestyle='--', label=f'Regression Line (Slope: {slope:.2f})')

plt.xlabel("Year")
plt.ylabel("Average Obesity Percentage")
plt.title("National Adult Obesity Trend with Linear Regression")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Linear regression analysis revealed a statistically significant increasing trend in national adult obesity prevalence over time (slope = 0.56, p < 0.001). This indicates that obesity rates have been rising steadily by approximately 0.56 percentage points per year.

## 3.2 Male vs Female

**Why this test is used:** The two-group test assesses whether the observed male–female mean difference is larger than expected from sampling variation.


In [ ]:
# Hypothesis questions: Is There a Difference in Obesity Between Males and Females?

analysis_df = clean_df[(clean_df['StratificationCategory1'] == 'Sex')].copy()
summary = analysis_df.groupby('Stratification1')['Data_Value'].agg(['mean', 'std', 'count']).round(2)

print("--- Obesity Prevalence Descriptive Statistics by Gender (%) ---")
print(summary)
print("\n")

# T-test
males_obesity = analysis_df[analysis_df['Stratification1'] == 'Male']['Data_Value']
females_obesity = analysis_df[analysis_df['Stratification1'] == 'Female']['Data_Value']

# The comparison is made based on the average obesity rate of each state/year
t_stat, p_val_t = stats.ttest_ind(males_obesity, females_obesity, nan_policy='omit')

print("--- T-Test Results ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val_t:.4f}")

if p_val_t < 0.05:
    print("Result: Statistically Significant difference in obesity prevalence between genders.")
else:
    print("Result: No significant difference in obesity prevalence.")

Statistical Finding: While female obesity prevalence ($\mu=31.13\%$) appears slightly higher than male prevalence ($\mu=30.80\%$), the difference is not statistically significant ($p = 0.1586$). This suggests that gender is not a primary determinant of obesity rates across the surveyed populations.

**Why this test is used:** The two-group test assesses whether the observed male–female mean difference is larger than expected from sampling variation.


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 6))

# Boxplot
plt.subplot(1, 2, 1)
sns.boxplot(x='Stratification1', y='Data_Value', data=analysis_df, palette='tab10')
plt.title('Distribution of Obesity Prevalence by Gender', fontsize=14, fontweight='bold')
plt.ylabel('Prevalence (%)', fontsize=12)
plt.xlabel('Gender', fontsize=12)

# Barplot
plt.subplot(1, 2, 2)
sns.barplot(x='Stratification1', y='Data_Value', data=analysis_df, palette='tab10', capsize=.1)
plt.title('Average Obesity Prevalence by Gender', fontsize=14, fontweight='bold')
plt.ylabel('Average Prevalence (%)', fontsize=12)
plt.xlabel('Gender', fontsize=12)

# P - Value
males = analysis_df[analysis_df['Stratification1'] == 'Male']['Data_Value']
females = analysis_df[analysis_df['Stratification1'] == 'Female']['Data_Value']
t_stat, p_val = stats.ttest_ind(males.dropna(), females.dropna())

plt.figtext(0.5, 0.01, f"Statistical Test: Independent t-test | P-value = {p_val:.4f} (Not Significant)",
            ha="center", fontsize=12, bbox={"facecolor":"#f0f0f0", "alpha":0.5, "pad":5})

plt.tight_layout()
plt.show()

## 3.3 Education Differences

**What this code does:** Reviews field cardinality to identify identifiers, categories, and columns that may need special treatment.


In [ ]:
import pandas as pd
from scipy.stats import f_oneway
import matplotlib.pyplot as plt
import seaborn as sns

# Load BRFSS obesity dataset if 'data' is not already defined
# This ensures the dataframe is available even if previous cells were not run.
if 'data' not in globals():
    data = pd.read_csv('Nutrition,_Physical_Activity,_and_Obesity_-_Behavioral_Risk_Factor_Surveillance_System_20260121.csv')
    data.columns = data.columns.str.strip().str.replace(r'\s+', '_', regex=True)
    # Re-apply type conversion and standardization as it was done previously for 'data'
    text_cols = [
        "LocationAbbr", "LocationDesc", "Datasource", "Class", "Topic", "Question",
        "Data_Value_Type", "Data_Value_Unit",
        "StratificationCategory1", "Stratification1",
        "Total", "Age(years)", "Education", "Sex", "Income", "Race/Ethnicity"
    ]
    for col in text_cols:
        if col in data.columns:
            data[col] = data[col].astype("string").str.strip()

    numeric_cols = [
        'YearStart', 'YearEnd',
        'Data_Value', 'Data_Value_Alt',
        'Low_Confidence_Limit', 'High_Confidence_Limit',
        'Sample_Size'
    ]

    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')

# Define the target obesity question (should be available from previous cells, but added here for robustness)
target_question = "Percent of adults aged 18 years and older who have obesity"

# 1. Research question
print("Research Question:")
print("Do adult obesity rates differ significantly across different education levels?\n")

print("Null Hypothesis (H0):")
print("The mean obesity prevalence is the same across all education levels.\n")

print("Alternative Hypothesis (H1):")
print("At least one education level has a different mean obesity prevalence.\n")

# 2. Prepare education dataset
# Ensure 'target_question' is defined from previous steps (cell '8t3OoM0pINso')
# Filter directly from the original 'data' DataFrame to ensure all relevant education data for the target question is included.
edu_df = data[
    (data['Question'] == target_question) &
    (data['StratificationCategory1'] == 'Education') &
    (data['Data_Value'].notna())
].copy()

print("Education dataset shape:", edu_df.shape)
print("Number of education levels:", edu_df["Stratification1"].nunique(), "\n")

# 3. Descriptive summary
edu_summary = (
    edu_df.groupby("Stratification1")["Data_Value"]
    .agg(["count", "mean", "std"])
    .sort_values("mean", ascending=False)
)

print("Obesity Prevalence Descriptive Statistics by Education Level (%):\n")
print(edu_summary, "\n")

# 4. Perform One-Way ANOVA
education_groups = edu_df.groupby("Stratification1")["Data_Value"]
groups = [group.values for name, group in education_groups]

anova_stat, anova_p = f_oneway(*groups)

print("One-Way ANOVA Result (Education Differences):")
print(f"F-statistic = {anova_stat:.4f}")
print(f"p-value = {anova_p:.6f}\n")

# 5. Hypothesis decision
alpha = 0.05

if anova_p < alpha:
    print("Conclusion:")
    print("Reject H0. There is a statistically significant difference in obesity prevalence across education levels.\n")
else:
    print("Conclusion:")
    print("Fail to reject H0. There is no statistically significant evidence that obesity prevalence differs across education levels.\n")

# 6. Visualization: Boxplot of Obesity Distribution by Education Level

# Define the order of education groups for better visualization
education_order = [
    "Less than high school",
    "High school graduate",
    "Some college or technical school",
    "College graduate"
]

plt.figure(figsize=(10, 6))
sns.boxplot(x="Stratification1", y="Data_Value", data=edu_df, order=education_order, palette="viridis")

plt.title("Obesity Distribution Across Education Levels", fontsize=16)
plt.xlabel("Education Level", fontsize=12)
plt.ylabel("Obesity Percentage", fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("The ANOVA test reveals whether there are statistically significant differences in adult obesity prevalence across different education levels. If the p-value is less than 0.05, it indicates that at least one education group has a significantly different mean obesity rate from the others. The boxplot visualization further illustrates the distribution and median obesity rates for each education category, visually supporting the statistical findings.")

## 3.4 Geographic Differences

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# 1. Research question

print("Research Question:")
print("Do adult obesity rates differ significantly across geographic locations (states/territories)?\n")

print("Null Hypothesis (H0):")
print("The mean obesity prevalence is the same across all geographic locations.\n")

print("Alternative Hypothesis (H1):")
print("At least one geographic location has a different mean obesity prevalence.\n")

**What this code does:** Reviews field cardinality to identify identifiers, categories, and columns that may need special treatment.


In [ ]:

# 2. Prepare geographic dataset

geo_df = clean_df[["LocationDesc", "YearStart", "Data_Value"]].dropna().copy()

print("Geographic dataset shape:", geo_df.shape)
print("Number of locations:", geo_df["LocationDesc"].nunique(), "\n")

# sample size per state
state_counts = geo_df["LocationDesc"].value_counts().sort_values(ascending=False)
print("Sample count by location (top 10):")
print(state_counts.head(10), "\n")


**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:

# 3. Descriptive summary

geo_summary = (
    geo_df.groupby("LocationDesc")["Data_Value"]
    .agg(["count", "mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

print("Top 10 locations with highest average obesity prevalence:")
print(geo_summary.head(10), "\n")

print("Top 10 locations with lowest average obesity prevalence:")
print(geo_summary.tail(10), "\n")

**Why this test is used:** One-way ANOVA evaluates whether average prevalence differs across more than two groups. A significant result is followed by practical interpretation and, where appropriate, post-hoc testing.


In [ ]:
from scipy.stats import levene, shapiro

# 4. Assumption checks

# 4a. Levene’s test for homogeneity of variance
groups = [
    grp["Data_Value"].values
    for _, grp in geo_df.groupby("LocationDesc")
    if len(grp) >= 2
]

levene_stat, levene_p = levene(*groups)
print("Levene’s Test for Homogeneity of Variance")
print(f"Statistic = {levene_stat:.4f}, p-value = {levene_p:.4f}\n")

# 4b. Normality check on ANOVA residuals
geo_df["state_mean"] = geo_df.groupby("LocationDesc")["Data_Value"].transform("mean")
geo_df["residual"] = geo_df["Data_Value"] - geo_df["state_mean"]

# Shapiro on a sample of residuals if too large
residual_sample = geo_df["residual"].dropna()
if len(residual_sample) > 5000:
    residual_sample = residual_sample.sample(5000, random_state=42)

shapiro_stat, shapiro_p = shapiro(residual_sample)
print("Shapiro-Wilk Test on Residuals")
print(f"Statistic = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}\n")

**Why this test is used:** One-way ANOVA evaluates whether average prevalence differs across more than two groups. A significant result is followed by practical interpretation and, where appropriate, post-hoc testing.


In [ ]:
from scipy.stats import f_oneway

# 5. One-Way ANOVA

anova_stat, anova_p = f_oneway(*groups)

print("One-Way ANOVA Result")
print(f"F-statistic = {anova_stat:.4f}")
print(f"p-value = {anova_p:.6f}\n")

alpha = 0.05
if anova_p < alpha:
    print("Conclusion:")
    print("Reject H0. There is a statistically significant difference in obesity prevalence across geographic locations.\n")
else:
    print("Conclusion:")
    print("Fail to reject H0. There is no statistically significant evidence that obesity prevalence differs across geographic locations.\n")

**Why this test is used:** One-way ANOVA evaluates whether average prevalence differs across more than two groups. A significant result is followed by practical interpretation and, where appropriate, post-hoc testing.


In [ ]:

# 7. Post-hoc test (Tukey HSD)

if anova_p < alpha:
    tukey = pairwise_tukeyhsd(
        endog=geo_df["Data_Value"],
        groups=geo_df["LocationDesc"],
        alpha=0.05
    )

    tukey_results = pd.DataFrame(
        data=tukey.summary().data[1:],
        columns=tukey.summary().data[0]
    )

    print("Tukey HSD Pairwise Comparison (first 20 rows):")
    print(tukey_results.head(20), "\n")

**Why this analysis matters:** Location averages highlight where prevention resources and deeper investigation may be most valuable.


In [ ]:

# 8. Visualization 1: Average obesity by state

state_avg = (
    geo_df.groupby("LocationDesc")["Data_Value"]
    .mean()
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 12))
state_avg.plot(kind="barh")
plt.xlabel("Average Obesity %")
plt.ylabel("State / Territory")
plt.title("Average Adult Obesity by Geographic Location")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Why this analysis matters:** Location averages highlight where prevention resources and deeper investigation may be most valuable.


In [ ]:

# 9. Visualization 2: Top 5 vs Bottom 5 trend comparison
top_5 = state_avg.sort_values(ascending=False).head(5).index
bottom_5 = state_avg.head(5).index

plt.figure(figsize=(10, 6))

for state in list(top_5) + list(bottom_5):
    state_data = (
        geo_df[geo_df["LocationDesc"] == state]
        .groupby("YearStart")["Data_Value"]
        .mean()
        .reset_index()
    )
    plt.plot(state_data["YearStart"], state_data["Data_Value"], marker='o', label=state)

plt.xlabel("Year")
plt.ylabel("Average Obesity Percentage")
plt.title("Geographic Trend Comparison: Highest vs Lowest Obesity Locations")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Why this test is used:** One-way ANOVA evaluates whether average prevalence differs across more than two groups. A significant result is followed by practical interpretation and, where appropriate, post-hoc testing.


In [ ]:

# 10. Short interpretation

print("Interpretation:")
print(
    "The geographic analysis evaluates whether adult obesity prevalence differs across states and territories. "
    "The ANOVA result indicates whether the observed variation in state-level obesity rates is statistically significant. "
    "If significant, the Tukey post-hoc test helps identify which locations differ from one another. "
    "The visualizations also show that some locations consistently report higher obesity prevalence than others over time, "
    "supporting the existence of geographic disparities in obesity outcomes."
)

## 3.5  Time Trend

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# 1. Research question

print("Research Question:")
print("Has adult obesity prevalence increased significantly over time?\n")

print("Null Hypothesis (H0):")
print("There is no increasing trend in obesity prevalence over time.\n")

print("Alternative Hypothesis (H1):")
print("Obesity prevalence shows a statistically significant increasing trend over time.\n")

**What this code does:** Reviews field cardinality to identify identifiers, categories, and columns that may need special treatment.


In [ ]:
# 2. Prepare trend dataset

trend_df = clean_df[["YearStart", "Data_Value"]].dropna().copy()

print("Trend dataset shape:", trend_df.shape)
print("Number of years:", trend_df["YearStart"].nunique(), "\n")

# Aggregate to national yearly mean
trend_yearly = (
    trend_df.groupby("YearStart")["Data_Value"]
    .mean()
    .reset_index()
)

print("Yearly trend preview:")
print(trend_yearly.head(), "\n")

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# 3. Descriptive summary

trend_summary = trend_yearly["Data_Value"].describe()

print("Trend Summary Statistics:")
print(trend_summary, "\n")

**Why this model is used:** Linear regression summarizes the average annual change and tests whether the observed time trend is statistically distinguishable from zero.


In [ ]:
# 4. Linear regression trend test

from scipy.stats import linregress

slope, intercept, r_value, p_value, std_err = linregress(
    trend_yearly["YearStart"],
    trend_yearly["Data_Value"]
)

print("Linear Regression Results:")
print(f"Slope = {slope:.4f}")
print(f"Intercept = {intercept:.4f}")
print(f"R-squared = {r_value**2:.4f}")
print(f"p-value = {p_value:.6f}\n")

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# 5. Hypothesis decision

alpha = 0.05

if p_value < alpha and slope > 0:
    print("Conclusion:")
    print("Reject H0. There is a statistically significant increasing trend in obesity prevalence over time.\n")
else:
    print("Conclusion:")
    print("Fail to reject H0. There is no statistically significant increasing trend in obesity prevalence.\n")

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# 6. Effect size

effect_size = r_value**2

print(f"Effect Size (R-squared) = {effect_size:.4f}\n")

**Why this step matters:** Confidence-interval width indicates estimate precision and helps prevent over-interpreting uncertain subgroup estimates.


In [ ]:
# 7. Visualization 1: Overall trend

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

plt.figure(figsize=(10,6))

# scatter points
sns.scatterplot(
    x=trend_yearly["YearStart"],
    y=trend_yearly["Data_Value"],
    s=80,
    color="steelblue",
    label="Observed yearly average"
)

# regression line with confidence interval
sns.regplot(
    x=trend_yearly["YearStart"],
    y=trend_yearly["Data_Value"],
    scatter=False,
    color="darkred",
    line_kws={"linewidth":2},
    label="Trend line"
)

plt.xlabel("Year", fontsize=12)
plt.ylabel("Average Obesity Prevalence (%)", fontsize=12)

plt.title(
    "Trend of Adult Obesity Prevalence in the United States (2011–2024)",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(trend_yearly["YearStart"])

plt.legend()

plt.tight_layout()

plt.show()

**Why this analysis matters:** Location averages highlight where prevention resources and deeper investigation may be most valuable.


In [ ]:
# 8. Visualization 2: State trend comparison

state_avg = (
    clean_df.groupby("LocationDesc")["Data_Value"]
    .mean()
    .sort_values(ascending=False)
)

top_states = state_avg.head(5).index
bottom_states = state_avg.tail(5).index

plt.figure(figsize=(10,6))

for state in list(top_states) + list(bottom_states):

    state_data = (
        clean_df[clean_df["LocationDesc"] == state]
        .groupby("YearStart")["Data_Value"]
        .mean()
        .reset_index()
    )

    plt.plot(
        state_data["YearStart"],
        state_data["Data_Value"],
        marker="o",
        label=state
    )

plt.xlabel("Year")
plt.ylabel("Average Obesity Percentage")
plt.title("Trend Comparison: Highest vs Lowest Obesity States")

plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")

plt.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()

plt.show()

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# 9. Short interpretation
print("""
Interpretation:

The trend analysis evaluates whether obesity prevalence has increased over time.

The linear regression results show a positive slope of approximately 0.56, meaning adult obesity prevalence increases by about 0.56 percentage points per year on average.

The p-value is less than 0.05, indicating that this upward trend is statistically significant.

The R-squared value of 0.9788 suggests that nearly 98% of the variation in yearly obesity prevalence is explained by the passage of time.

These results provide strong statistical evidence that adult obesity prevalence in the United States has increased steadily over the observed period.
""")